# 03 — Quantum Similarity
Kernel computation, sanity checks, classical baseline, CKA, KPCA.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)
SEED = 42
np.random.seed(SEED)
for d in ['../outputs/figures','../outputs/tables','../outputs/kernels',
          '../outputs/logs','../data/processed','../data/raw']:
    os.makedirs(d, exist_ok=True)
print("Setup done.")

Setup done.


## Sample N=150 (seed=42)

In [2]:
from src.data import load_wine_quality, scale_features_to_pi, get_deterministic_sample, WINE_QUANTUM_FEATURES, WINE_TARGET_COL
df = load_wine_quality('../data/raw/winequality-red.csv')
N = 150
idx, df_sample = get_deterministic_sample(df, N, seed=42)
print(f"Selected {N} observations. Indices range: {idx.min()} – {idx.max()}")
np.save('../outputs/tables/sample_indices.npy', idx)

X_sample = df_sample[WINE_QUANTUM_FEATURES].values
X_sample_scaled = scale_features_to_pi(X_sample)
quality_sample = df_sample[WINE_TARGET_COL].values
print(f"X_sample_scaled.shape: {X_sample_scaled.shape}")

Loaded wine quality: 1599 rows × 12 columns
Selected 150 observations. Indices range: 23 – 1587
X_sample_scaled.shape: (150, 4)


## Build Quantum Kernel

In [3]:
from src.quantum import build_zz_feature_map, build_quantum_kernel, compute_kernel_matrix
fm = build_zz_feature_map(n_qubits=4, reps=2, entanglement='linear')
qkernel = build_quantum_kernel(fm)
K_q, elapsed = compute_kernel_matrix(qkernel, X_sample_scaled)
print(f"K_q shape: {K_q.shape}, elapsed: {elapsed:.1f}s")
np.save('../outputs/kernels/K_quantum.npy', K_q)
print("Saved K_quantum.npy")

ZZFeatureMap: 4 qubits, reps=2, entanglement=linear
  Circuit depth: 19, Parameters: 4
FidelityQuantumKernel built (StatevectorSampler + ComputeUncompute).
Computing quantum kernel matrix (150x150) ...
  Done in 109.1s. K.shape=(150, 150)
K_q shape: (150, 150), elapsed: 109.1s
Saved K_quantum.npy


## Sanity Checks

In [4]:
from src.metrics import kernel_sanity_check
results_q = kernel_sanity_check(K_q, 'K_quantum')


── Sanity Check: K_quantum ──
  name: K_quantum
  shape: (150, 150)
  symmetry_error: 1.6653345369377348e-16
  diagonal_mean: 1.0010918925002947
  diagonal_min: 1.0000224192117664
  nan_count: 0
  inf_count: 0
  min_eigenvalue: -4.661015271472168e-16
  is_psd: True
  kernel_mean: 0.08118932336202025
  kernel_std: 0.11320838848529158


## Classical RBF Kernel Baseline

In [5]:
from sklearn.preprocessing import StandardScaler
from src.classical import compute_classical_kernel
X_sample_std = StandardScaler().fit_transform(df_sample[WINE_QUANTUM_FEATURES].values)
K_c = compute_classical_kernel(X_sample_std)
np.save('../outputs/kernels/K_classical.npy', K_c)
results_c = kernel_sanity_check(K_c, 'K_classical')
print("Saved K_classical.npy")


── Sanity Check: K_classical ──
  name: K_classical
  shape: (150, 150)
  symmetry_error: 2.220446049250313e-16
  diagonal_mean: 1.0
  diagonal_min: 1.0
  nan_count: 0
  inf_count: 0
  min_eigenvalue: -2.936081218758718e-17
  is_psd: True
  kernel_mean: 0.28098073037632637
  kernel_std: 0.25987858251973567
Saved K_classical.npy


## Kernel Heatmaps

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, K, title in zip(axes, [K_q, K_c], ['Quantum Kernel', 'Classical RBF Kernel']):
    im = ax.imshow(K, cmap='viridis', aspect='auto')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Sample index'); ax.set_ylabel('Sample index')
    plt.colorbar(im, ax=ax)
plt.suptitle('Kernel Heatmaps (N=150)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/07b_kernel_heatmaps.png', dpi=120, bbox_inches='tight')
plt.close()
print("Saved 07b_kernel_heatmaps.png")

Saved 07b_kernel_heatmaps.png


## Centered Kernel Alignment (CKA)

In [7]:
from src.metrics import centered_kernel_alignment
cka = centered_kernel_alignment(K_q, K_c)
print(f"Classical–Quantum CKA: {cka:.6f}")
import csv
os.makedirs('../outputs/tables', exist_ok=True)
with open('../outputs/tables/research_summary.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['metric', 'value'])
    w.writerow(['classical_quantum_cka', f'{cka:.6f}'])
print("Saved research_summary.csv")

Classical–Quantum CKA: 0.248383
Saved research_summary.csv


## Quantum Kernel PCA

In [8]:
from src.classical import run_kernel_pca
kpca_emb = run_kernel_pca(K_q)
fig, ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(kpca_emb[:,0], kpca_emb[:,1], c=quality_sample, cmap='viridis', alpha=0.7, s=25)
plt.colorbar(sc, ax=ax, label='Quality')
ax.set_xlabel('KPCA-1'); ax.set_ylabel('KPCA-2')
ax.set_title('Quantum Kernel PCA (N=150, color=quality – interpretation only)')
plt.tight_layout()
plt.savefig('../outputs/figures/07_quantum_kpca.png', dpi=120, bbox_inches='tight')
plt.close()
np.save('../outputs/tables/kpca_emb.npy', kpca_emb)
print("Saved 07_quantum_kpca.png")
print("Notebook 03 complete.")

Saved 07_quantum_kpca.png
Notebook 03 complete.
